#PROYECTO - ANÁLISIS DE CRÍTICAS DE PELICULAS FILMAFFINITY EN ESPAÑOL

El sentimiento de análisis (sentiment analysis en inglés) es una subdisciplina del campo procesamiento del lenguaje natural (NLP en inglés) bastante común y que nos permite identificar la opinión emocional que hay detrás de un texto, es decir, si es positivo, negativo o neutro.

Como podéis imaginaros, esta técnica tiene innumerables aplicaciones debido a su gran utilidad. Por ejemplo, el análisis de sentimientos podría ayudar a una empresa a entender el sentimiento que provoca su marca, producto o servicio aplicándolo a los tweets en los que son mencionados.

Para este tutorial nos hemos basado en el ejemplo práctico que aparece en el capítulo 8 del fantástico libro Python Machine Learning, donde utilizan 50.000 críticas de películas altamente polares del sitio web IMDb para construir un clasificador capaz de identificar si una crítica de una película tiene un sentimiento positivo o negativo.

Vamos a utilizar un conjunto de datos consistente en 4.800 críticas de usuarios y sus puntuaciones correspondientes de la página web de filmaffinity. Filmaffinity es una web de recomendación de películas y series donde los usuarios pueden escribir y compartir sus propias reseñas de cada película junto con una valoración numérica que puede ir del 1 al 10.

---
# Librerías a utilizar

In [3]:
import nltk
nltk.download('stopwords') #descargar los recursos de nltk

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
#import mysql.connector
import pandas as pd
import numpy as np
#import json
import re
#import eli5
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.tokenize import ToktokTokenizer
#from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix

In [6]:
# Cargamos el fichero con las críticas y su puntuación
#df = pd.read_csv("/content/sample_data/proyecto.csv", encoding='utf8')

df = pd.read_table('/content/sample_data/proyecto.csv', sep='\|\|', header=0, engine='python')
df.sample(5)

# Mostramos las primeras 5 observaciones
df.head()

,film_name,gender,film_avg_rate,review_rate,review_title,review_text
0,Ocho apellidos vascos,Comedia,"6,0",3.0,OCHO APELLIDOS VASCOS...Y NINGÚN NOMBRE PROPIO,La mayor virtud de esta película es su existen...
1,Ocho apellidos vascos,Comedia,"6,0",2.0,El perro verde,"No soy un experto cinéfilo, pero pocas veces m..."
2,Ocho apellidos vascos,Comedia,"6,0",2.0,Si no eres de comer mierda... no te comas esta...,Si no eres un incondicional del humor estilo T...
3,Ocho apellidos vascos,Comedia,"6,0",2.0,Aida: The movie,"No sé qué está pasando, si la gente se deja ll..."
4,Ocho apellidos vascos,Comedia,"6,0",2.0,UN HOMBRE SOLO (Julio Iglesias 1987),"""Pero cuando amanece,y me quedo solo,siento en..."


In [7]:
# Creamos una variable con el sentimiento
# Si puntuación > 6 -> 1
# Si puntuación < 5 -> 0
df['sentiment'] = np.where(df['review_rate'] > 6, 1, 0)

# Eliminamos las variables nota y url
df.drop(columns=["film_name","gender", "film_avg_rate", "review_rate", "review_title"], inplace=True)

df.head()

,review_text,sentiment
0,La mayor virtud de esta película es su existen...,0
1,"No soy un experto cinéfilo, pero pocas veces m...",0
2,Si no eres un incondicional del humor estilo T...,0
3,"No sé qué está pasando, si la gente se deja ll...",0
4,"""Pero cuando amanece,y me quedo solo,siento en...",0


---
# Limpieza y preprocesamiento

In [8]:
tokenizer = ToktokTokenizer()
STOPWORDS = set(stopwords.words("spanish"))
stemmer = SnowballStemmer("spanish")

def limpiar_texto(texto):
    """
    Función para realizar la limpieza de un texto dado.
    """
    # Eliminamos los caracteres especiales
    texto = re.sub(r'\W', ' ', str(texto))
    # Eliminado las palabras que tengo un solo caracter
    texto = re.sub(r'\s+[a-zA-Z]\s+', ' ', texto)
    # Sustituir los espacios en blanco en uno solo
    texto = re.sub(r'\s+', ' ', texto, flags=re.I)
    # Eliminar numeros
    texto = re.sub(r'[0-9]+', '', texto)
    # Convertimos textos a minusculas
    texto = texto.lower()
    return texto

In [9]:
def filtrar_stopword_digitos(tokens):
    """
    Filtra stopwords y digitos de una lista de tokens.
    """
    return [token for token in tokens if token not in STOPWORDS
            and not token.isdigit()]

In [10]:
def stem_palabras(tokens):
    """
    Reduce cada palabra de una lista dada a su raíz.
    """
    return [stemmer.stem(token) for token in tokens]

In [11]:
def tokenize(texto):
    """
    Método encargado de realizar la limpieza y preprocesamiento de un texto
    """
    text_cleaned = limpiar_texto(texto)
    tokens = [word for word in tokenizer.tokenize(text_cleaned) if len(word) > 1]
    tokens = filtrar_stopword_digitos(tokens)
    stems = stem_palabras(tokens)
    return stems

In [12]:
df_1 = df.copy()
df_val1=df_1[df_1['sentiment']==1]
df_val1

,review_text,sentiment
5,La llegada de Rafa a Euskadi es como ponerse a...,1
21,Divertidísima película la de Emilio Martínez-L...,1
23,"¿Cuantas comedias americanas de ""éxito"" he vis...",1
27,"Ahora que el cine está más barato, merece la p...",1
31,Tanto criticar el despilfarro cultural en el c...,1
...,...,...
8591,Qué decir de un director tan acertado como Alb...,1
8593,"Buenísimo.., salgo encantado. Digna de ver, de...",1
8594,Me ha parecido una obra maestra de principio a...,1
8596,Lleva un tiempo el cine español sorprendiéndom...,1


In [13]:
#*****************************************************
df_val0=df_1[df_1['sentiment']==0]
#df_val0=df_val3.sample(n = 400)
df_val0

,review_text,sentiment
0,La mayor virtud de esta película es su existen...,0
1,"No soy un experto cinéfilo, pero pocas veces m...",0
2,Si no eres un incondicional del humor estilo T...,0
3,"No sé qué está pasando, si la gente se deja ll...",0
4,"""Pero cuando amanece,y me quedo solo,siento en...",0
...,...,...
8597,Esto es lo que consideramos bueno?Pagar una en...,0
8599,"Me esperaba mucho, pero que mucho, más.Guión m...",0
8600,"De mal cuerpo como sensación al finalizar, de ...",0
8601,Los que han añadido comentarios os lo han dich...,0


In [14]:
df_final=pd.concat([df_val1,df_val0])
df_final

,review_text,sentiment
5,La llegada de Rafa a Euskadi es como ponerse a...,1
21,Divertidísima película la de Emilio Martínez-L...,1
23,"¿Cuantas comedias americanas de ""éxito"" he vis...",1
27,"Ahora que el cine está más barato, merece la p...",1
31,Tanto criticar el despilfarro cultural en el c...,1
...,...,...
8597,Esto es lo que consideramos bueno?Pagar una en...,0
8599,"Me esperaba mucho, pero que mucho, más.Guión m...",0
8600,"De mal cuerpo como sensación al finalizar, de ...",0
8601,Los que han añadido comentarios os lo han dich...,0


import numpy as np
from google.colab import autoviz

def value_plot(df, y, figscale=1):
  from matplotlib import pyplot as plt
  df[y].plot(kind='line', figsize=(8 * figscale, 4 * figscale), title=y)
  plt.gca().spines[['top', 'right']].set_visible(False)
  plt.tight_layout()
  return autoviz.MplChart.from_current_mpl_state()

chart = value_plot(df_final, *['sentiment'], **{})
chart

import numpy as np
from google.colab import autoviz

def histogram(df, colname, num_bins=20, figscale=1):
  from matplotlib import pyplot as plt
  df[colname].plot(kind='hist', bins=num_bins, title=colname, figsize=(8*figscale, 4*figscale))
  plt.gca().spines[['top', 'right',]].set_visible(False)
  plt.tight_layout()
  return autoviz.MplChart.from_current_mpl_state()

chart = histogram(df_final, *['sentiment'], **{})
chart

In [17]:
df_final[["review_text"]].head()

,review_text
5,La llegada de Rafa a Euskadi es como ponerse a...
21,Divertidísima película la de Emilio Martínez-L...
23,"¿Cuantas comedias americanas de ""éxito"" he vis..."
27,"Ahora que el cine está más barato, merece la p..."
31,Tanto criticar el despilfarro cultural en el c...


In [21]:
tokenizer = ToktokTokenizer()

texto = df_final[["review_text"]].head()[0:1].to_numpy()[0][0]
texto_limpio = limpiar_texto(texto)
tokens = [word for word in tokenizer.tokenize(texto_limpio) if len(word) > 1]
tokens = filtrar_stopword_digitos(tokens)
stems = stem_palabras(tokens)
stems

['lleg',
 'raf',
 'euskadi',
 'pon',
 'ver',
 'apell',
 'vasc',
 'ira',
 'buen',
 'pues',
 'afortun',
 'pelicul',
 'lej',
 'mied',
 'lej',
 'convencional',
 'tem',
 'tabu',
 'hac',
 'pued',
 'parec',
 'previs',
 'tipic',
 'comedi',
 'romant',
 'enred',
 'ambient',
 'grand',
 'contrast',
 'conoc',
 'cultur',
 'lengu',
 'clim',
 'etc',
 'etc',
 'encim',
 'agreg',
 'tradicion',
 'tan',
 'dispar',
 'vam',
 'dec',
 'coñ',
 'tipic',
 'comedi',
 'tont',
 'romant',
 'enred',
 'consab',
 'tont',
 'siempr',
 'amig',
 'funcion',
 'funcion',
 'result',
 'quier',
 'descubr',
 'sab',
 'quier',
 'nadi',
 'salg',
 'teoriz',
 'conflict',
 'vasc',
 'quier',
 'cuent',
 'lej',
 'teor',
 'quier',
 'calor',
 'herrik',
 'tabern',
 'chist',
 'tont',
 'escen',
 'previs',
 'gent',
 'call',
 'mar',
 'entonc',
 'pued',
 'dar',
 'cuent',
 'just',
 'mejor',
 'cualid',
 'pelicul',
 'verd',
 'much',
 'import',
 'absolut',
 'sencillez',
 'ningun',
 'joy',
 'cin',
 'esplend',
 'revel',
 'autent',
 'bombaz',
 'pues',
 '

# Dividimos los datos en train y test

In [23]:
X = df_final.review_text
y = df_final.sentiment
# SEPARAR EN 70% Y 30%
X_train_txt, X_test_txt, y_train, y_test = train_test_split(X, y, test_size=0.33,
                                                    stratify=y, random_state=42)

In [24]:
X_test_txt

7270    Se le dio mucha caña al Orfanato por considera...
7280    La película, para quien no la haya visto, es b...
3548    Desencanta y aún más si se ha leído a Perez Re...
5442    Recuerdo ver esta película un día de enero con...
1716    Cómo influyen las malas críticas. Aunque en el...
                              ...                        
4688    Es una película muy divertida y fantástica.Se ...
1181    Desde el inicio hasta el final, el orfanato es...
3946    Nos llega esta “Celda 211” después del éxito d...
8119    Una de las peores películas que he visto en lo...
1022    No era de extrañar que, tras el insólito y des...
Name: review_text, Length: 2839, dtype: object

In [25]:
# Comprobamos que train y test contienen el mismo porcentaje de cada clase en y
print(len(y_train)/len(y)) #verificamos que la data de entrenamiento sea de 67%
print(len(y_test)/len(y)) #verificamos que la data de test sea de 33%

#comprobamos balance de datos
print(len(y_train[y_train==1])/len(y_train)) #verificamos que los datos esten balanceados por categoria dentro del entrenamiento
print(len(y_test[y_test==1])/len(y_test)) #verificamos que los datos esten balanceados por categoria dentro del test

0.6699988376147855
0.3300011623852145
0.45558639833448994
0.45579429376541036


In [26]:
tfidf = TfidfVectorizer(
    tokenizer=tokenize,
    max_features=20000)

tfidf.fit(X_train_txt)

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:528: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


TfidfVectorizer(max_features=20000,
                tokenizer=<function tokenize at 0x791550d71240>)

In [27]:
X_train = tfidf.transform(X_train_txt)
X_test = tfidf.transform(X_test_txt)
print(X_train.shape)
print(X_test.shape)

(5764, 20000)
(2839, 20000)


In [28]:
X_train

<5764x20000 sparse matrix of type '<class 'numpy.float64'>'
	with 505758 stored elements in Compressed Sparse Row format>

#CODIFICACIÓN

In [29]:
from keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train_txt)
MODO='binary' #fidf,freq,binary,count,tfidf
prueba_X_train = tokenizer.texts_to_matrix(X_train_txt, mode=MODO)

In [30]:
print(tokenizer.word_index.items())

dict_items([('de', 1), ('que', 2), ('la', 3), ('y', 4), ('en', 5), ('el', 6), ('a', 7), ('no', 8), ('es', 9), ('un', 10), ('una', 11), ('los', 12), ('se', 13), ('con', 14), ('lo', 15), ('por', 16), ('película', 17), ('del', 18), ('las', 19), ('más', 20), ('pero', 21), ('como', 22), ('para', 23), ('su', 24), ('me', 25), ('al', 26), ('muy', 27), ('esta', 28), ('todo', 29), ('o', 30), ('si', 31), ('cine', 32), ('ha', 33), ('sin', 34), ('bien', 35), ('le', 36), ('historia', 37), ('ya', 38), ('sus', 39), ('ni', 40), ('está', 41), ('nos', 42), ('son', 43), ('tiene', 44), ('ser', 45), ('porque', 46), ('este', 47), ('hay', 48), ('mejor', 49), ('ver', 50), ('te', 51), ('tan', 52), ('mi', 53), ('hace', 54), ('mucho', 55), ('algo', 56), ('eso', 57), ('nada', 58), ('poco', 59), ('gran', 60), ('sobre', 61), ('aunque', 62), ('personajes', 63), ('cuando', 64), ('películas', 65), ('todos', 66), ('español', 67), ('he', 68), ('guión', 69), ('también', 70), ('final', 71), ('menos', 72), ('puede', 73), ('

In [31]:
prueba_X_train

array([[0., 1., 1., ..., 0., 0., 0.],
       [0., 1., 1., ..., 0., 0., 0.],
       [0., 1., 1., ..., 0., 0., 0.],
       ...,
       [0., 1., 1., ..., 0., 0., 0.],
       [0., 1., 1., ..., 0., 0., 0.],
       [0., 1., 1., ..., 1., 1., 1.]])

In [32]:
print(X_train[:3])

  (0, 19645)	0.18205705702877095
  (0, 19620)	0.10071105472581451
  (0, 19415)	0.18559484953124333
  (0, 19369)	0.06429348021506545
  (0, 19162)	0.0869831614912786
  (0, 18494)	0.07417474496123178
  (0, 18091)	0.07006542258066598
  (0, 17509)	0.06589645886997694
  (0, 17209)	0.05489969785845686
  (0, 17105)	0.09927370226317808
  (0, 17005)	0.0983522375740612
  (0, 16764)	0.07851493208938096
  (0, 16273)	0.10137259408124184
  (0, 15390)	0.08984497576368784
  (0, 15041)	0.07965789471167335
  (0, 14603)	0.13377799816432503
  (0, 14013)	0.03551869833046281
  (0, 14007)	0.10187846866180557
  (0, 13868)	0.1495373801035758
  (0, 13738)	0.06369776316372278
  (0, 13226)	0.16603795414895317
  (0, 12075)	0.060423088840770064
  (0, 12024)	0.09238111343960546
  (0, 11824)	0.13187638720659658
  (0, 11707)	0.09922176294222662
  :	:
  (2, 12102)	0.07059482746103954
  (2, 11579)	0.06335766344324846
  (2, 11427)	0.10298917469832482
  (2, 11284)	0.07895728951411383
  (2, 10040)	0.24231853731079137
  (2, 

# DESARROLLO E IMPLEMENTACION DEL MODELO

Ha llegado la hora de entrenar nuestro modelo de regresión logística para clasificar las reseñas de películas. Para ello, empleamos una estrategia grid search para realizar una búsqueda exhaustiva evaluando todas las combinaciones de parámetros con el objetivo de encontrar el mejor modelo.

Grid search es una técnica de optimización de hiperparámetros en el que se prueban todas las combinaciones posibles. A continuación, los modelos se evalúan mediante validación cruzada y se considera que el modelo con mayor accuracy (en nuestro caso) es el mejor.

Para utilizar esta estrategia, scikit learn nos proporciona el objeto GridSearchCV al que le debemos pasar como input el modelo de predicción y un diccionario con los parámetros (llamado search space) a optimizar. Este diccionario debe contener los nombres de parámetros como claves y las listas de ajustes de parámetros a probar como valores.

En nuestro caso queremos probar distintos valores del parámetro C, que añade regularización al modelo para reducir el overfitting, y el parámetro penalty para probar distintos tipos de regularización (Lasso y Ridge en nuestro caso).

In [33]:
# Diccionario con nombres de parámetros como claves y listas
# de ajustes de parámetros a probar como valores
parameters = {'penalty':('l1', 'l2'), 'C':[100, 10, 1.0, 0.1, 0.01]}

# Modelo de Regresión logistica
lr = LogisticRegression(random_state=42, solver='liblinear')

In [34]:
# Objecto KFold para dividir un conjunto de datos en N bloques
cv = KFold(n_splits=4, shuffle=True, random_state=42)

In [35]:
# GridSearchCV para la búsqueda de los mejores parámetros
clf = GridSearchCV(lr, parameters,
                   scoring='accuracy',
                   cv=cv,
                   refit=True,
                   verbose=2,
                   n_jobs=-1)

In [42]:
clf.fit(X_train, y_train)

Fitting 4 folds for each of 10 candidates, totalling 40 fits


GridSearchCV(cv=KFold(n_splits=4, random_state=42, shuffle=True),
             estimator=LogisticRegression(random_state=42, solver='liblinear'),
             n_jobs=-1,
             param_grid={'C': [100, 10, 1.0, 0.1, 0.01],
                         'penalty': ('l1', 'l2')},
             scoring='accuracy', verbose=2)

In [43]:
print('Mejor combinación de parámetros: %s ' % clf.best_params_)
print('CV Accuracy: %.3f' % clf.best_score_)

Mejor combinación de parámetros: {'C': 1.0, 'penalty': 'l2'} 
CV Accuracy: 0.795


In [45]:
best_clf = clf.best_estimator_
print('Test Accuracy: %.3f' % best_clf.score(X_test, y_test))

Test Accuracy: 0.805


<img src="https://empresas.blogthinkbig.com/wp-content/uploads/2020/09/matriz.jpg" alt="Simply Easy Learning" width="400"
         height="200">

In [46]:
y_pred = best_clf.predict(X_test)
confusion_matrix(y_test, y_pred)

array([[1317,  228],
       [ 327,  967]])

In [47]:
!pip install eli5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for eli5: filename=eli5-0.13.0-py2.py3-none-any.whl size=107719 sha256=edd4dbd1ed8761bbe50be056d234624ebfc583fc52ad06a1a6ba327a33dc11fc
  Stored in directory: /root/.cache/pip/wheels/b8/58/ef/2cf4c306898c2338d51540e0922c8e0d6028e07007085c0004
Successfully built eli5


In [48]:
import eli5

eli5.show_weights(estimator=best_clf,
                  feature_names= list(tfidf.get_feature_names_out()),
                 top=(10, 10))

Weight?,Feature
+2.716,perfect
+2.667,genial
+2.639,excelent
+2.396,recomend
+2.390,gust
+2.302,gran
+2.242,mantien
+2.239,maravill
+2.086,disfrut
+2.036,encant


# Predicción de nuevos datos

In [64]:
tokenizer = ToktokTokenizer()

opinion=["la pelicula me desagrada"]
X_prueba = tfidf.transform(opinion)
print(X_prueba.shape)


(1, 20000)


In [65]:
best_clf.predict(X_prueba)

array([0])

---
